# 📘 Notebook 10 · 强化学习用于交易

> 强化学习是量化里"看起来最有前途"的方向，**也是最容易被坑的方向**。这一节我们既给代码，也告诉你哪些地方不要踩。

**学完你会：**

- 理解 MDP、Q-Learning、Policy Gradient 的核心
- 用 DQN 写一个简单的择时 agent
- 用 PPO 做组合配置 / 仓位管理
- 知道金融 RL 的三大陷阱
- 看清做市、执行、组合三种 RL 应用的真实状态

**预计 8-12 小时。**

**前置**：Notebook 09 + PyTorch 基础。

## 1. 为什么需要 RL，监督学习不够吗

监督学习的逻辑：**预测 → 决策**（先预测涨跌，再决定买卖）

问题：
- **预测和决策耦合**：预测准 ≠ 决策好（如成本、容量、组合约束）
- **不能优化长期目标**：MSE 损失和"年化收益 / 回撤"是两回事
- **不能处理交互**：你的买卖会影响价格（冲击成本），监督学习不知道

强化学习直接优化**最终目标**：

$$\max_\pi \mathbb{E}\left[\sum_{t=0}^T \gamma^t r_t \right]$$

把"决策"和"环境反馈"放在一个循环里学。

### 1.1 RL 在量化里的三大应用

| 应用 | 状态 $s$ | 动作 $a$ | 奖励 $r$ |
|---|---|---|---|
| **择时** | 价量特征 | 买 / 卖 / 持有 | PnL |
| **组合管理** | 多资产因子 | 各资产权重 | 夏普 / Sortino |
| **执行算法** | 订单簿状态 | 切单大小 / 时机 | 价差 + 冲击 |
| **做市** | 库存 + 订单簿 | 报价价差 | 价差 - 库存风险 |

**应用难度排序**（从易到难）：执行 > 组合管理 > 择时 > 做市

## 2. MDP 与三大算法

### 2.1 MDP（马尔可夫决策过程）

五元组：$(S, A, P, R, \gamma)$

- $S$：状态空间
- $A$：动作空间
- $P(s' | s, a)$：转移概率
- $R(s, a)$：奖励函数
- $\gamma \in [0,1)$：折扣因子

**目标**：找一个策略 $\pi(a|s)$，最大化期望回报。

### 2.2 三大算法家族

**Value-Based**：学价值函数 $Q(s, a)$，决策时取 argmax

- Q-Learning（表格）
- DQN（神经网络）
- 适合**离散动作**（买/卖/持有）

**Policy-Based**：直接学策略 $\pi(a|s)$

- REINFORCE
- PPO、TRPO
- 适合**连续动作**（持仓比例）

**Actor-Critic**：混合两者

- A2C / A3C
- SAC、DDPG、TD3

### 2.3 Q-Learning 公式

$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[r + \gamma \max_{a'} Q(s', a') - Q(s, a)\right]$$

直觉：当前 Q 值 += 学习率 × (TD 误差)

### 2.4 DQN 改进

- **Experience Replay**：把 (s, a, r, s') 存进 buffer，随机采样训练（打破样本相关性）
- **Target Network**：另一份固定权重的 Q 网络，避免 bootstrap 不稳定
- **Double DQN**：用主网络选 action，target 网络估值
- **Dueling DQN**：分别学 $V(s)$ 和 $A(s, a)$

## 3. 一个 DQN 择时 Agent

我们让一个 agent 学"什么时候持有 ETF、什么时候空仓"。**简化场景**，主要是教学。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
    print('需要 PyTorch')

# ---------- 一个简单的市场模拟器 ----------
class StockEnv:
    """
    State: 过去 N 日的收益率 + 当前持仓 (0/1)
    Action: 0=空仓, 1=持有
    Reward: 当日 PnL (持仓收益)
    """
    def __init__(self, prices, window=20, tx_cost=0.0005):
        self.prices = prices.astype(np.float32)
        self.returns = np.diff(np.log(prices)).astype(np.float32)
        self.window = window
        self.tx_cost = tx_cost
        self.reset()

    def reset(self):
        self.t = self.window
        self.pos = 0
        self.equity = 1.0
        self.history = []
        return self._state()

    def _state(self):
        # 状态：过去 window 日收益 + 当前持仓
        ret_window = self.returns[self.t - self.window : self.t]
        return np.concatenate([ret_window, [self.pos]]).astype(np.float32)

    def step(self, action):
        # action: 0 / 1
        old_pos = self.pos
        new_pos = action

        # 当日收益 = 持仓 × 当日实际收益
        daily_ret = self.returns[self.t] if self.t < len(self.returns) else 0
        pnl = new_pos * daily_ret

        # 交易成本：持仓变化才扣
        cost = self.tx_cost if new_pos != old_pos else 0
        reward = pnl - cost

        self.pos = new_pos
        self.equity *= (1 + reward)
        self.history.append({'t': self.t, 'pos': new_pos, 'ret': daily_ret,
                             'reward': reward, 'equity': self.equity})

        self.t += 1
        done = self.t >= len(self.returns) - 1
        return self._state(), reward, done


# 构造一个有趋势的价格序列
np.random.seed(42)
n_days = 1000
trend = np.zeros(n_days)
trend[200:400]   =  0.0015
trend[600:800]   = -0.0012
trend[100:200]   =  0.0008
returns = trend + np.random.randn(n_days) * 0.015
prices = 100 * np.cumprod(1 + returns)

env = StockEnv(prices, window=20)
state = env.reset()
print(f'状态形状: {state.shape}  (前 20 个是收益率，最后 1 个是持仓)')

In [ ]:
if HAS_TORCH:
    # ---------- DQN 网络 ----------
    class QNet(nn.Module):
        def __init__(self, n_state, n_action=2, hidden=64):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(n_state, hidden), nn.ReLU(),
                nn.Linear(hidden, hidden), nn.ReLU(),
                nn.Linear(hidden, n_action)
            )
        def forward(self, x):
            return self.net(x)

    # ---------- Experience Replay ----------
    from collections import deque
    import random

    class ReplayBuffer:
        def __init__(self, capacity=10000):
            self.buf = deque(maxlen=capacity)
        def push(self, *args):
            self.buf.append(args)
        def sample(self, batch_size):
            batch = random.sample(self.buf, batch_size)
            return [np.array(x) for x in zip(*batch)]
        def __len__(self):
            return len(self.buf)

    # ---------- 训练 ----------
    n_state = 21
    q_net = QNet(n_state)
    target_net = QNet(n_state)
    target_net.load_state_dict(q_net.state_dict())
    opt = optim.Adam(q_net.parameters(), lr=1e-3)
    buf = ReplayBuffer(10000)

    n_episodes = 30
    gamma = 0.95
    epsilon = 1.0
    eps_min = 0.05
    eps_decay = 0.95
    batch_size = 64

    rewards_per_ep = []
    for ep in range(n_episodes):
        state = env.reset()
        total_reward = 0
        while True:
            # ε-greedy
            if np.random.rand() < epsilon:
                action = np.random.choice(2)
            else:
                with torch.no_grad():
                    qs = q_net(torch.from_numpy(state).unsqueeze(0))
                    action = int(qs.argmax(1).item())

            next_state, reward, done = env.step(action)
            buf.push(state, action, reward, next_state, float(done))
            state = next_state
            total_reward += reward

            # 从 buffer 采样训练
            if len(buf) >= batch_size:
                ss, aa, rr, ss2, dd = buf.sample(batch_size)
                ss   = torch.from_numpy(ss.astype(np.float32))
                aa   = torch.from_numpy(aa.astype(np.int64))
                rr   = torch.from_numpy(rr.astype(np.float32))
                ss2  = torch.from_numpy(ss2.astype(np.float32))
                dd   = torch.from_numpy(dd.astype(np.float32))

                q_sa = q_net(ss).gather(1, aa.unsqueeze(1)).squeeze(1)
                with torch.no_grad():
                    q_max = target_net(ss2).max(1).values
                    target = rr + gamma * q_max * (1 - dd)
                loss = ((q_sa - target) ** 2).mean()
                opt.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(q_net.parameters(), 1.0)
                opt.step()

            if done:
                break

        epsilon = max(eps_min, epsilon * eps_decay)
        rewards_per_ep.append(env.equity - 1)

        # 定期同步 target net
        if (ep+1) % 5 == 0:
            target_net.load_state_dict(q_net.state_dict())

        if (ep+1) % 5 == 0:
            print(f'Ep {ep+1:2d}  ε={epsilon:.3f}  终值收益={env.equity-1:+.4%}')

In [ ]:
if HAS_TORCH:
    # ---------- 用学好的策略做一次回测 ----------
    state = env.reset()
    actions = []
    while True:
        with torch.no_grad():
            qs = q_net(torch.from_numpy(state).unsqueeze(0))
            action = int(qs.argmax(1).item())
        actions.append(action)
        state, r, done = env.step(action)
        if done: break

    history = pd.DataFrame(env.history)

    # 画图
    fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    axes[0].plot(prices, color='gray', alpha=0.7, label='价格')
    in_pos = np.array([h['pos'] for h in env.history])
    pos_idx = np.where(in_pos == 1)[0] + env.window
    axes[0].scatter(pos_idx, prices[pos_idx], s=5, color='green', alpha=0.4, label='持仓中')
    axes[0].set_title('DQN 学到的择时策略')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(history['equity'], color='darkred', label='策略净值')
    axes[1].plot(np.cumprod(1 + returns[env.window:env.window+len(history)]),
                  color='steelblue', alpha=0.7, label='买入持有')
    axes[1].set_title('累计净值'); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()

    print(f'\n策略最终净值: {env.equity:.4f}')
    print(f'买持基准:    {np.prod(1 + returns[env.window:]):.4f}')

**注意：**

1. 这是**最简化的 DQN**——真实策略要复杂很多
2. **模型可能过拟合**——同一数据训练 + 测试，回测结果不可靠
3. **正确做法是用 walk-forward**：训练前 60% 数据，测试后 40%
4. **奖励设计是核心**：直接用 PnL 容易学到激进策略；可改成 "PnL - λ·|Δposition|" 或 Sharpe

### 3.1 RL 择时的真实评估

在真实 A 股 / 美股数据上，DQN 择时**几乎一定输给简单移动平均策略**。原因：

- 信噪比低，RL 容易过拟合训练区间
- 收益和动作的关联弱，credit assignment 困难
- 探索（exploration）在金融上代价大（亏真金白银）

**结论**：择时 RL 是教学例子，不是生产环境的现实选择。

### 🎯 练习

三个练习：

1. 用 walk-forward：训练集 600 天，测试集 400 天，看 DQN 在样本外能不能仍然赚钱
2. 把动作扩展到 3 个：空仓 / 半仓 / 满仓。看 agent 能不能学到"高波动期减仓"
3. 改写奖励为 `Sharpe(过去 20 日)` 而不是当日 PnL，看策略稳定性如何

In [ ]:
# 在这里写你的答案


## 4. PPO 用于组合配置：更现实的应用

### 4.1 PPO 公式速览

Proximal Policy Optimization 是 OpenAI 2017 提出的，**目前 RL 应用最广的算法**。

核心思想：**限制新策略 vs 旧策略的变化幅度**，避免一次更新跳太远导致崩溃。

PPO Clipped 目标：

$$L^{CLIP}(\theta) = \mathbb{E}_t\left[\min(r_t(\theta)\hat A_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon)\hat A_t)\right]$$

- $r_t(\theta) = \pi_\theta(a_t|s_t) / \pi_{\theta_{old}}(a_t|s_t)$
- $\hat A_t$：advantage 估计（GAE）
- $\epsilon$ 一般取 0.2

### 4.2 组合管理为什么适合 RL

| 特点 | 监督学习 | RL |
|---|---|---|
| 直接优化目标 | MSE/CE | Sharpe / Sortino / Calmar |
| 多期决策 | 单步预测 | 多步累计回报 |
| 约束（杠杆、敞口） | 后处理 | 直接编码到 action space |
| 交易成本 | 后处理 | 直接 reward 里扣 |

**PPO 组合管理的真实工业应用：** AQR、Bridgewater 都有研究 PPO 用于宏观资产配置（股 / 债 / 商品权重）。

In [ ]:
# 一个简化的 PPO 组合管理
# 状态：3 个资产过去 N 日特征
# 动作：3 个资产的目标权重（连续，∑=1，∈[0,1]）
# 奖励：当期组合 Sharpe

if HAS_TORCH:
    class PortfolioEnv:
        def __init__(self, returns_matrix, window=10):
            self.returns = returns_matrix.astype(np.float32)
            self.n_assets = returns_matrix.shape[1]
            self.window = window
            self.reset()

        def reset(self):
            self.t = self.window
            self.weights = np.ones(self.n_assets) / self.n_assets   # 初始等权
            self.equity_history = [1.0]
            self.reward_history = []
            return self._state()

        def _state(self):
            past = self.returns[self.t - self.window : self.t].flatten()
            return np.concatenate([past, self.weights]).astype(np.float32)

        def step(self, new_weights):
            # 归一化让权重为合法概率分布
            new_weights = np.clip(new_weights, 0, 1)
            new_weights = new_weights / (new_weights.sum() + 1e-8)
            # 当期组合收益
            ret = (new_weights * self.returns[self.t]).sum()
            tx = 0.0005 * np.abs(new_weights - self.weights).sum()
            net_ret = ret - tx

            # Reward: 用当期收益作为奖励（也可以用最近 20 日 Sharpe）
            reward = net_ret * 100   # 放大方便训练

            self.weights = new_weights
            self.equity_history.append(self.equity_history[-1] * (1 + net_ret))
            self.reward_history.append(net_ret)
            self.t += 1
            done = self.t >= len(self.returns) - 1
            return self._state(), reward, done

    # 模拟 3 个资产
    np.random.seed(0)
    n_days = 1500
    ret_a = np.random.randn(n_days) * 0.012      # 股
    ret_b = np.random.randn(n_days) * 0.005      # 债
    ret_c = np.random.randn(n_days) * 0.020      # 商品
    R = np.stack([ret_a, ret_b, ret_c], axis=1)
    env = PortfolioEnv(R, window=10)

    s = env.reset()
    print(f'状态形状: {s.shape}  (30 个过去收益 + 3 个当前权重)')

In [ ]:
if HAS_TORCH:
    # ---------- PPO 网络（Actor + Critic 共享 backbone）----------
    class ActorCritic(nn.Module):
        def __init__(self, n_state, n_action, hidden=64):
            super().__init__()
            self.backbone = nn.Sequential(
                nn.Linear(n_state, hidden), nn.Tanh(),
                nn.Linear(hidden, hidden), nn.Tanh(),
            )
            # Actor: 输出每个权重的均值（用 Beta 分布或 Softmax）
            self.actor_mean = nn.Linear(hidden, n_action)
            self.actor_log_std = nn.Parameter(torch.zeros(n_action) - 0.5)
            # Critic
            self.critic = nn.Linear(hidden, 1)

        def forward(self, x):
            h = self.backbone(x)
            mean = torch.tanh(self.actor_mean(h)) * 0.5 + 0.5   # 限制到 [0, 1]
            log_std = self.actor_log_std.expand_as(mean).clamp(-3, 2)
            std = torch.exp(log_std)
            value = self.critic(h)
            return mean, std, value

    n_state = 30 + 3
    n_action = 3
    ac = ActorCritic(n_state, n_action)
    opt_ac = optim.Adam(ac.parameters(), lr=3e-4)

    # ---------- 简化的 PPO 主循环 ----------
    def compute_gae(rewards, values, dones, gamma=0.99, lam=0.95):
        """广义优势估计"""
        gae, returns = 0, []
        next_value = 0
        for r, v, d in zip(reversed(rewards), reversed(values), reversed(dones)):
            delta = r + gamma * next_value * (1 - d) - v
            gae = delta + gamma * lam * (1 - d) * gae
            returns.insert(0, gae + v)
            next_value = v
        return np.array(returns)

    n_episodes = 20
    clip_eps = 0.2
    final_equities = []

    for ep in range(n_episodes):
        # 收集一个 episode 的数据
        states, actions, log_probs, rewards, values, dones = [], [], [], [], [], []
        s = env.reset()
        while True:
            with torch.no_grad():
                s_t = torch.from_numpy(s).unsqueeze(0)
                mean, std, value = ac(s_t)
                dist = torch.distributions.Normal(mean, std)
                a = dist.sample()
                lp = dist.log_prob(a).sum(-1)
            a_np = a.numpy().flatten()
            s2, r, done = env.step(a_np)

            states.append(s); actions.append(a_np); log_probs.append(lp.item())
            rewards.append(r); values.append(value.item()); dones.append(float(done))

            s = s2
            if done: break

        # 算 GAE 优势
        returns = compute_gae(rewards, values, dones)
        advantages = returns - np.array(values)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        # 转 Tensor
        s_t   = torch.from_numpy(np.array(states))
        a_t   = torch.from_numpy(np.array(actions))
        lp_t  = torch.from_numpy(np.array(log_probs))
        adv_t = torch.from_numpy(advantages.astype(np.float32))
        ret_t = torch.from_numpy(returns.astype(np.float32))

        # K 次更新
        for _ in range(4):
            mean, std, value = ac(s_t)
            dist = torch.distributions.Normal(mean, std)
            new_lp = dist.log_prob(a_t).sum(-1)
            ratio = torch.exp(new_lp - lp_t)
            surr1 = ratio * adv_t
            surr2 = torch.clamp(ratio, 1-clip_eps, 1+clip_eps) * adv_t
            actor_loss = -torch.min(surr1, surr2).mean()
            critic_loss = ((value.squeeze() - ret_t) ** 2).mean()
            entropy = dist.entropy().sum(-1).mean()
            loss = actor_loss + 0.5 * critic_loss - 0.01 * entropy

            opt_ac.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(ac.parameters(), 0.5)
            opt_ac.step()

        final_equities.append(env.equity_history[-1])
        if (ep+1) % 5 == 0:
            print(f'Ep {ep+1:2d}  终值={env.equity_history[-1]:.4f}  '
                  f'mean rew/step={np.mean(rewards):.4f}')

**这段 PPO 代码是给你理解机制的简化版**。**真实工业用 RL** 需要：

- 大量并行环境（A3C / IMPALA / R2D2）
- 更长的训练（百万步以上）
- 仔细的奖励工程（避免 reward hacking）
- 用 [Ray RLlib](https://docs.ray.io/en/latest/rllib/) 或 [Stable-Baselines3](https://github.com/DLR-RM/stable-baselines3) 这种成熟框架

下面的练习用 Stable-Baselines3 替代手撕，更接近生产。

## 5. 金融 RL 三大陷阱

### 5.1 陷阱 1：过拟合 + 看似很好的回测

**症状**：训练曲线漂亮，测试曲线一样漂亮，**实盘崩盘**

**原因**：
- 监督学习的过拟合 + RL 的过度探索 + 金融数据的非平稳
- 状态包含未来信息（数据泄露）
- 测试数据与训练数据时间相邻

**对策**：
- 严格 walk-forward
- Out-of-sample 至少 12 个月
- 多个市场环境验证（牛 / 熊 / 震荡）

### 5.2 陷阱 2：奖励设计错误（Reward Hacking）

**例子**：奖励 = 当日 PnL
- Agent 学到：每天买入然后卖出，频繁交易
- 看起来胜率高，实际被手续费吃掉一半

**例子**：奖励 = 累计收益
- Agent 学到：极致激进，把杠杆开满
- 一次回撤直接清仓

**正确做法**：
- 奖励 = PnL - λ·|Δ position| - μ·max_drawdown
- 用 "差异 Sharpe"（Sharpe 增量）作为 reward
- 多个奖励组合，权重调优

### 5.3 陷阱 3：探索成本极高

监督学习的探索"免费"（不会损失任何东西）。RL 的探索意味着**真的下单亏钱**。

**对策**：
- 先在 simulator 里训练（off-policy）
- 上 paper trading 6 个月以上
- 实盘上极小资金 + 加大探索阻力（Boltzmann 而不是 ε-greedy）

### 5.4 一个反面教材

2018-2019 年很多论文宣称 "RL 在加密货币上跑赢市场 100%+"。

事后发现：
- 测试集是 2017 年单边牛市 → 任何"满仓持有"策略都能赢
- 训练集和测试集时间不分割
- 没有交易成本

**任何宣称 RL "稳定跑赢"的论文，先检查这三个问题。**

## 6. RL 在工业中的真实应用

### 6.1 应用 1：订单执行（最成熟）

**问题**：要卖 1000 万股，怎么切单使滑点最小？

**经典做法**：TWAP / VWAP / Implementation Shortfall

**RL 做法**：根据订单簿实时状态调整切单速度

**真实使用**：高盛 / JP Morgan 等大行已部署生产环境。论文：Nevmyvaka et al. 2006 "Reinforcement Learning for Optimized Trade Execution"

### 6.2 应用 2：做市

**问题**：买卖双边报价 + 控制库存风险

**经典做法**：Avellaneda-Stoikov 解析解

**RL 做法**：状态包含库存 + 订单簿，动作是 bid/ask 价差

**真实使用**：HRT、Jane Street 等据传都有 RL 做市的实验，但**没有在生产环境替代经典模型**。原因：经典模型理论清晰、可解释、稳定。

### 6.3 应用 3：组合管理

**问题**：多资产权重分配

**经典做法**：Markowitz / Risk Parity / Black-Litterman

**RL 做法**：以历史数据为环境，学习动态调仓策略

**真实使用**：很多论文，**主流私募基本不用 RL 做主动管理**。

### 6.4 总结：哪些方向值得投入

| 方向 | 真实价值 | 难度 |
|---|---|---|
| 订单执行 RL | 高 ✅ | 中 |
| 做市 RL | 中 | 极高 |
| 组合管理 RL | 中 | 中 |
| 择时 RL | 低 ❌ | 中 |
| 多策略动态分配 RL | 高 ✅ | 高 |

**实习面试想吹 RL**，重点讲**执行**或**多策略分配**——这是真实工业最看重的，比"我做了一个 RL 择时"高级一截。

## 7. 推荐资源

**入门：**
- [Sutton & Barto《Reinforcement Learning: An Introduction》](http://incompleteideas.net/book/the-book-2nd.html) 免费圣经
- David Silver 的 UCL RL 课程（YouTube）

**深度：**
- OpenAI Spinning Up（PPO / SAC 代码注释清晰）
- Berkeley CS285 课程

**金融 RL 框架：**
- [FinRL](https://github.com/AI4Finance-Foundation/FinRL)：开源金融 RL，可上手
- [Stable-Baselines3](https://github.com/DLR-RM/stable-baselines3)：通用 RL，金融可用
- [Ray RLlib](https://docs.ray.io/en/latest/rllib/)：分布式 RL

**关键论文：**
- Nevmyvaka et al. 2006: 订单执行 RL 鼻祖
- Cartea et al. 2015: 做市的最优控制理论
- Buehler et al. 2019 "Deep Hedging"：DL 用于对冲

---

**下一节：Notebook 11 · LLM 与另类数据** — 我们看怎么用大模型分析新闻、研报、卫星图像，和现在头部私募在做什么。